# Model Training & Batch Transform DAG
This is the core ML pipeline. We pull data from our **Feature Store**, train a Random Forest on SageMaker infrastructure, and execute a **Batch Transform** job to generate predictions.

In [1]:
!pip install "sagemaker<3.0.0" awswrangler

import os
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
import awswrangler as wr
import time

# Setup
role = sagemaker.get_execution_role()
sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "aai-540-group6/nyc-collisions-ml"

print(f"Executing as Role: {role}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Executing as Role: arn:aws:iam::594126158441:role/LabRole


### 1. Retrieve Data from Feature Store
Run an Athena query to grab the engineered features from our Offline Store.

In [2]:
# feature group name
fg_name = "collisions-07-05-51"

fg = FeatureGroup(name=fg_name, sagemaker_session=sess)
query = fg.athena_query()
db = query.database
tbl = query.table_name

sql = f'SELECT * FROM "{db}"."{tbl}"'
print(f"Pulling data via: {sql}")

df = wr.athena.read_sql_query(sql, database=db)
print(f"Retrieved {df.shape[0]} rows.")

Pulling data via: SELECT * FROM "sagemaker_featurestore"."collisions_07_05_51_1780811507"


2026-06-07 06:19:10,405	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 906997760 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=1.88gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-07 06:19:11,595	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Retrieved 50000 rows.


### 2. Prepare the Splits
We need a 3-way split: Training, Validation, and a Batch inference set (without the target label).

In [3]:
features = ['borough', 'month', 'hour', 'is_rush_hour', 'is_weekend', 'cause_category', 'vehicle_type']
target = 'target'
id_col = 'collision_id'

# Random split
rand = np.random.rand(len(df))
train_idx = rand < 0.8
val_idx = (rand >= 0.8) & (rand < 0.9)
batch_idx = rand >= 0.9

df_train = df[train_idx][features + [target]]
df_val = df[val_idx][features + [target]]
df_batch = df[batch_idx][features + [id_col]] # IDs kept for joining

print(f"Splits -> Train: {len(df_train)}, Val: {len(df_val)}, Batch: {len(df_batch)}")

Splits -> Train: 39909, Val: 5017, Batch: 5074


### 3. Upload to S3

In [4]:
os.makedirs('data_splits', exist_ok=True)
df_train.to_csv('data_splits/train.csv', index=False, header=True)
df_val.to_csv('data_splits/val.csv', index=False, header=True)
df_batch.to_csv('data_splits/batch.csv', index=False, header=True)

s3_train = sess.upload_data('data_splits/train.csv', bucket, f"{prefix}/train")
s3_val = sess.upload_data('data_splits/val.csv', bucket, f"{prefix}/validation")
s3_batch = sess.upload_data('data_splits/batch.csv', bucket, f"{prefix}/batch")

### 4. Training (SKLearn Estimator)
Kick off a training job on a dedicated ml.m5.xlarge instance.

In [5]:
from sagemaker.sklearn.estimator import SKLearn

est = SKLearn(
    entry_point='sagemaker_train.py',
    source_dir='../src',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    framework_version='1.2-1',
    py_version='py3',
    hyperparameters={'n-estimators': 100, 'max-depth': 12}
)

est.fit({'train': s3_train})
print("Model fit complete.")

INFO:sagemaker:Creating training-job with name: sagemaker-scikit-learn-2026-06-07-06-19-15-291


2026-06-07 06:19:17 Starting - Starting the training job.

.

.


2026-06-07 06:19:32 Starting - Preparing the instances for training.

...

.

.


2026-06-07 06:21:16 Training - Training image download completed. Training in progress.
2026-06-07 06:21:16 Uploading - Uploading generated training model/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-06-07 06:21:09,846 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-06-07 06:21:09,851 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-06-07 06:21:09,854 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-06-07 06:21:09,869 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-06-07 06:21:10,158 sagemaker-training-toolkit INFO     Installing de


2026-06-07 06:21:24 Failed - Training job failed


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:14                                                                                   │
│                                                                                                  │
│   11 │   hyperparameters={'n-estimators': 100, 'max-depth': 12}                                  │
│   12 )                                                                                           │
│   13                                                                                             │
│ ❱ 14 est.fit({'train': s3_train})                                                                │
│   15 print("Model fit complete.")                                                                │
│   16                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:171 in wrapper  │
│                                                                                                  │
│   168 │   │   │   │   │   caught_ex = e                                                          │
│   169 │   │   │   │   finally:                                                                   │
│   170 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 171 │   │   │   │   │   │   raise caught_ex                                                    │
│   172 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   173 │   │   │   else:                                                                          │
│   174 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:142 in wrapper  │
│                                                                                                  │
│   139 │   │   │   │   start_timer = perf_counter()                                               │
│   140 │   │   │   │   try:                                                                       │
│   141 │   │   │   │   │   # Call the original function                                           │
│ ❱ 142 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   143 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   144 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   145 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                            

### 5. Batch Inferences (I/O Joining)
Execute the transform job and join predictions with their IDs.

In [ ]:
trans = est.transformer(
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f"s3://{bucket}/{prefix}/predictions",
    assemble_with='Line'
)

trans.transform(
    data=s3_batch,
    content_type='text/csv', 
    split_type='Line',
    input_filter='$[1:]', # Send only features to model
    join_source='Input',
    output_filter='$[0,-1]' # Output ID and result
)

trans.wait()
print("Predictions ready in S3.")

### 6. Final Audit
Download and check the results.

In [ ]:
res = wr.s3.read_csv(f"s3://{bucket}/{prefix}/predictions/batch.csv.out", header=None)
res.columns = ['id', 'pred']
display(res.head())